# Project 03: Gibbs Sampling (Random Algorithm)

In [30]:
import numpy as np
import random
import seqlogo
from data_readers import *
from motif_ops import *
from seq_ops import *

---
## Implement Gibbs Sampler


Gibbs sampling is a MCMC approach to identify enrichments. Here we will implement a method to identify motifs from a set of regions. 

Important considerations:
- We will need to score each sequence with a PWM using the `score_kmer()` or `score_sequence()` functions
    - You will need to investigate into the help documentation and libraries to identify how best to use these functions.
- These sites are often not strand-specific and so both scores on the negative as well as positive strand should be considered
- To select a random sequence, use `random.randint()` or `numpy.random.randint()`
- To select a new position $m$ (as defined below) use `random.choices()` or `numpy.random.choice()`  

Assumptions: 
- We know $k$ as the length of expected motif
- Each sequence contains the motif



```
GibbsMotifFinder(DNA, k-length)
    random pick of k-length sequences from each line of DNA as Motifs
    for j ← 1 to 10000 or Motifs stops changing
        i ← Random(N) where N is number of DNA entries
        PWM ← PWM constructed from all Motifs except for Motifi
        Motifi ← select position m from PWM-scored k-mers in DNAi in probabilistic fashion from score distribution
    return PFM
```

Probability of choosing position $m = \frac{A_{m}}{\sum_{l}A_{l}}$ for positions $l$ in DNAi


**Note:** I have also added a function to `motif_ops.py` that will calculate the information content of your motifs. This is useful to observe the progression of your Gibbs sampler as well as a measure of convergence. You can use this function as `IC = pfm_ic(pfm)`. You should expect a slow increase of IC until it plateaus such as in the plot below from your lecture slides:

<center><img src='figures/Gibbs_Sampling.png'/ width=600px></center>

---
# Project Start

In [31]:
def GibbsMotifFinder(seqs, k, seed=None):
    """
    Function to find a pfm from a list of strings using a Gibbs sampler

    Args:
        seqs (str list): a list of sequences, not necessarily in same lengths
        k (int): the length of motif to find
        seed (int, default=None): seed for np.random

    Returns:
        pfm (numpy array): dimensions are 4xlength
    """
    # Use rng to make random samples/selections/numbers
    random.seed(seed)
    rng = np.random.default_rng(seed)

    # Only sequences at least k long can contain a k-mer
    seqs = [s.upper() for s in seqs if len(s) >= k] # check for inconsistencies in the sequencing data
    n = len(seqs)  # number of DNA entries

    # Initialization
    # Select a random position in each sequence as the initial 'motif'
    motifs = []
    for seq in seqs:
        start = rng.integers(0, len(seq) - k + 1)   # from the start of the seq to the end of the seq, because seq lengths can vary
        motifs.append(seq[start:start + k])

    # Iteration
    # Note on Convergence (from lecture slides): stop if the IC has not changed in the past 100 iterations
    window = 100
    ic_history = [pfm_ic(build_pfm(motifs, k))] # run the first pfm_ic (viewed in lecture video)

    for iteration in range(10000):
        # Choose a random sequence, i = index
        i = rng.integers(n)

        # Recalculate the PWM, omitting the chosen sequence
        not_chosen = motifs[:i] + motifs[i + 1:]
        pwm = build_pwm(build_pfm(not_chosen, k))

        # Score every possible k-mer in sequence i against the PWM, on both the fwd and rev strands
        fwd_strand = [seqs[i][m:m + k] for m in range(len(seqs[i]) - k + 1)] # m = position of the motif
        rev_strand = [reverse_complement(kmer) for kmer in fwd_strand]
        both_strands = fwd_strand + rev_strand # concatenate the probabilities of both strands
        scores = np.array([score_kmer(kmer, pwm) for kmer in both_strands]) # construct in one array


        # PWM scores are log2 odds (includes 0s, negatives and positives) -> convert to positive numbers.
        odds = 2 ** scores
        probability = odds / odds.sum()

        # Choose the new motif based on the score distribution
        m = rng.choice(len(both_strands), p=probability) # m = motif position
        motifs[i] = both_strands[m]   # rev-strands are stored as reverse complements

        # Convergence check on information content
        ic_history.append(pfm_ic(build_pfm(motifs, k)))
        if len(ic_history) > window and np.allclose(ic_history[-window - 1:], ic_history[-1]):
            break

    return build_pfm(motifs, k)


In [32]:
# Here we test your Gibbs sampler.
# You do not need to edit this or the section below. This is the Driver program

#read promoters, store in a list of strings
seq_file="data/GCF_000009045.1_ASM904v1_genomic.fna.gz"
gff_file="data/GCF_000009045.1_ASM904v1_genomic.gff.gz"

seqs = []

for name, seq in get_fasta(seq_file): # For each entry in our FASTA file
    for gff_entry in get_gff(gff_file): # For each entry in our GFF file
        if gff_entry.type == 'CDS': # If this is a coding sequence
            promoter_seq = get_seq(seq, gff_entry.start, gff_entry.end, gff_entry.strand, 50) # Extract 50 bp as a promoter
    
            """
            Because the gibbs sampling assumption is broken in just using promoters,
            and because it takes very long time to randomly progress through so many
            regions, for this example we will pre-filter for sequences that all contain
            part of the shine-dalgarno motif:
            """
            if "AGGAGG" in promoter_seq:
                seqs.append(promoter_seq)

FileNotFoundError: [Errno 2] No such file or directory: 'data/GCF_000009045.1_ASM904v1_genomic.fna.gz'

---
# Driver Program
Don't change any of the code here. If you have completed the project by following the coding by contract, the following code should work.

In [ ]:
# Run the gibbs sampler:
promoter_pfm = GibbsMotifFinder(seqs,10 )

# Plot the final pfm that is generated: 
seqlogo.seqlogo(seqlogo.CompletePm(pfm = promoter_pfm.T))

---
# Challenge Yourself
Now that you potentially have a working skeleton of the project, try to work with the new NRF1 ChIP-seq data: `data/nrf1_gibbs.fa`.

Upon inspection, you will see that it is just a FASTA file. Since it is a FASTA file, you already have an example in the code above on how to access FASTA files. It is up to you to do the correct data ingest with it, though.

---
# NRF1 Driver Program
Modify the code below and test your Gibbs Sampler.

In [33]:
# Here we test your Gibbs sampler.
# You do not need to edit this or the section below. This is the Driver program

#read promoters, store in a list of strings
nrf1_file="data/nrf1_gibbs.fa"

nrf1_peaks = []

# Data ingest of nrf1_peaks:
# Each FASTA entry is one 75 bp ChIP-seq peak; we only need the sequence
for name, seq in get_fasta(nrf1_file):
    seq = seq.upper() # Checks that the sequence is all in uppercase
    if "GCATGC" in seq: # Chose a random motif
        nrf1_peaks.append(seq)

# Take a reproducible random subset so each peak is resampled many times
random.seed(2)
nrf1_peaks = random.sample(nrf1_peaks, 500)

# Run the gibbs sampler:
nrf1_pfm = GibbsMotifFinder(nrf1_peaks, 10)

# Plot the final pfm that is generated: 
seqlogo.seqlogo(seqlogo.CompletePm(pfm = nrf1_pfm.T))

OSError: Could not find Ghostscript on path. There should be either a gs executable or a gswin32c.exe on your system's path